# Import danych do warstwy RAW

Pobieramy oryginalne pliki i zapisujemy je razem z metadanymi pobrania (URL, data, rozmiar, SHA-256). Źródła: GIOŚ (pomiary i metadane stacji) oraz Open-Meteo (pogoda).

Kod pobierania jest w `scripts/download_raw.py` i `scripts/download_openmeteo.py`.

## 1. Kontrola zależności

Korzystamy z istniejącego środowiska. Notebooki nie instalują zależności.

In [1]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "requests", "jupyterlab", "ipykernel", "matplotlib", "duckdb",
    "polars", "pyarrow", "fastexcel", "scikit-learn",
    "nbclient", "nbformat",
]
missing = []
for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Brak bibliotek: " + ", ".join(missing)
        + ". Zainstaluj requirements.txt we własnym terminalu."
    )


requests: 2.34.2
jupyterlab: 4.6.4
ipykernel: 7.4.0
matplotlib: 3.11.2
duckdb: 2.0.0
polars: 2.0.0
pyarrow: 26.0.0
fastexcel: 0.21.0
scikit-learn: 1.9.1
nbclient: 0.11.0
nbformat: 5.11.1


## 2. Lokalizacja projektu

Notebook można uruchomić z katalogu głównego projektu albo z katalogu `notebooks`.

In [2]:
from pathlib import Path
import json
import platform

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.pipeline_config import YEARS, PERIOD, START, END, HOURS
CONFIG_PATH = ROOT / 'config' / 'sources.json'
RAW_DIR = ROOT / 'data' / 'raw'
SCRIPT_PATH = ROOT / 'scripts' / 'download_raw.py'

if not CONFIG_PATH.is_file() or not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        'Uruchom notebook z katalogu projektu lub notebooks.'
    )

RAW_DIR.mkdir(parents=True, exist_ok=True)

exe = Path(sys.executable).resolve()
interpreter = exe.relative_to(ROOT).as_posix() if exe.is_relative_to(ROOT) else exe.name
print('Projekt:', ROOT.name)
print('Interpreter:', interpreter)
print('Python:', platform.python_version())


Projekt: gios-bigdata
Interpreter: .venv/Scripts/python.exe
Python: 3.13.1


## 3. Konfiguracja źródeł

Plik `config/sources.json` zawiera adresy plików GIOŚ. Wpisy z polem `year` trafiają do `data/raw/<rok>/`, a wpisy z polem `dataset` do `data/raw/<dataset>/`.

Źródło danych: GIOŚ - EKOINFONET (https://powietrze.gios.gov.pl/pjp/archives).

In [3]:
config = json.loads(
    CONFIG_PATH.read_text(encoding='utf-8-sig')
)

print(json.dumps(config, ensure_ascii=False, indent=2))

{
  "sources": [
    {
      "year": 2023,
      "url": "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/564",
      "filename": "pomiary_2023.zip"
    },
    {
      "year": 2024,
      "url": "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582",
      "filename": "pomiary_2024.zip"
    },
    {
      "year": 2025,
      "url": "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/644",
      "filename": "pomiary_2025.zip"
    },
    {
      "dataset": "stacje",
      "url": "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/643",
      "filename": "metadane_stacji.xlsx"
    }
  ]
}


## 4. Import danych GIOŚ do RAW

Pobieramy lata z config/sources.json (2023–2025) i metadane stacji. Pliki z poprawną sumą SHA-256 są pomijane.

In [4]:
from scripts import download_raw
download_raw.main()

2026-11-25 18:04:19,749 | INFO | Plik istnieje i jest poprawny — pomijam: pomiary_2023.zip


2026-11-25 18:04:19,778 | INFO | Plik istnieje i jest poprawny — pomijam: pomiary_2024.zip


2026-11-25 18:04:19,808 | INFO | Plik istnieje i jest poprawny — pomijam: pomiary_2025.zip


2026-11-25 18:04:19,810 | INFO | Plik istnieje i jest poprawny — pomijam: metadane_stacji.xlsx


2026-11-25 18:04:19,810 | INFO | Import do RAW zakończony.


[{'provider': 'GIOŚ',
  'layer': 'raw',
  'dataset': '2023',
  'year': 2023,
  'source_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/564',
  'resolved_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/564',
  'filename': 'pomiary_2023.zip',
  'downloaded_at_utc': '2026-11-25T16:00:53.432699+00:00',
  'size_bytes': 71451204,
  'sha256': '6de04165db2429586f8106911653b54e700759eb8bca476789031baa742e54ab',
  'content_type': 'application/x-zip-compressed'},
 {'provider': 'GIOŚ',
  'layer': 'raw',
  'dataset': '2024',
  'year': 2024,
  'source_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582',
  'resolved_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582',
  'filename': 'pomiary_2024.zip',
  'downloaded_at_utc': '2026-10-22T20:16:58.879638+00:00',
  'size_bytes': 49482718,
  'sha256': '571dfa56866388c2904284ca6029bbf6016af3905b95bcacc5b3b6f6fa2d00e1',
  'content_type': 'application/zip'},
 {'provider': 'GIOŚ',
  'layer': 'raw',

## 5. Pogoda — Open-Meteo

Pobieramy pełne lata UTC dla sumy stacji PM2.5 ze wszystkich roczników. Brak pomiarów stacji w danym roku pozostanie NULL w tabeli cech. Źródło: Open-Meteo Historical Weather API, CC BY 4.0. Między pobraniami są 2 sekundy przerwy. Każdy plik jest sprawdzany pod kątem pełnej siatki godzin.

In [5]:
from scripts import download_openmeteo
openmeteo_results = download_openmeteo.main()

2026-11-25 18:04:21,100 | INFO | Plik istnieje i jest poprawny — pomijam: DsChojnowMalMOB.json


2026-11-25 18:04:21,119 | INFO | Plik istnieje i jest poprawny — pomijam: DsDusznSportMOB.json


2026-11-25 18:04:21,137 | INFO | Plik istnieje i jest poprawny — pomijam: DsJelGorOgin.json


2026-11-25 18:04:21,155 | INFO | Plik istnieje i jest poprawny — pomijam: DsKlodzSzkol.json


2026-11-25 18:04:21,173 | INFO | Plik istnieje i jest poprawny — pomijam: DsKudowaSzkoMOB.json


2026-11-25 18:04:21,190 | INFO | Plik istnieje i jest poprawny — pomijam: DsLubanLudowMOB.json


... (372 outputs left out)

2026-11-25 18:04:28,373 | INFO | Plik istnieje i jest poprawny — pomijam: ZpSwinMatejkMOB.json


2026-11-25 18:04:28,395 | INFO | Plik istnieje i jest poprawny — pomijam: ZpSzczAndrze.json


2026-11-25 18:04:28,416 | INFO | Plik istnieje i jest poprawny — pomijam: ZpSzczPilsud.json


Stacje z pogodą: 381


## 6. Podsumowanie lokalnych danych RAW

Liczba i rozmiar pobranych plików w każdym folderze.

In [6]:
records = []

for metadata_path in sorted(
    RAW_DIR.rglob('*.metadata.json')
):
    metadata = json.loads(
        metadata_path.read_text(encoding='utf-8')
    )
    data_path = metadata_path.parent / metadata['filename']
    if data_path.is_file():
        metadata['folder'] = metadata_path.parent.relative_to(RAW_DIR).as_posix()
        records.append(metadata)

total_bytes = sum(item['size_bytes'] for item in records)

print('Liczba plików:', len(records))
print(f'Łączny rozmiar: {total_bytes / 1_000_000:.2f} MB')
print()

summary = {}

for item in records:
    folder = summary.setdefault(
        item['folder'],
        {'zrodlo': item['provider'], 'pliki': 0, 'bajty': 0},
    )
    folder['pliki'] += 1
    folder['bajty'] += item['size_bytes']

for name, info in summary.items():
    print(
        f"{name:18} {info['zrodlo']:12} "
        f"plików: {info['pliki']:4}   {info['bajty'] / 1_000_000:7.2f} MB"
    )

Liczba plików: 385
Łączny rozmiar: 327.22 MB

2023               GIOŚ         plików:    1     71.45 MB
2024               GIOŚ         plików:    1     49.48 MB
2025               GIOŚ         plików:    1     50.35 MB
openmeteo/2023     Open-Meteo   plików:  127     51.73 MB
openmeteo/2024     Open-Meteo   plików:  127     51.90 MB
openmeteo/2025     Open-Meteo   plików:  127     51.75 MB
stacje             GIOŚ         plików:    1      0.56 MB


## 7. Zakres dalszej analizy

Analizujemy rzeczywiste dane 2023–2025. Aktualne liczby i rozmiary plików są w podsumowaniu powyżej. Zachowujemy wszystkie wskaźniki poza depozycją. Celem jest godzinowe PM2.5; brak pomiaru nie jest zastępowany zerem.

Następnie uruchom notebooki 02 i 03.